# 03 - Inference, Benchmarking, And Local Deploy

Test the trained Drive model and document the local folder expected by Streamlit.


In [1]:
# ============================================================
# CONFIG
# ============================================================
PROJECT_ROOT = "/content/drive/MyDrive/vsl-recognition"
METADATA_DIR = f"{PROJECT_ROOT}/metadata"
DRIVE_MODEL_DIR = f"{PROJECT_ROOT}/models/videomae_olympic_best"
LOCAL_MODEL_DIR = "models/videomae_olympic_best"
NUM_FRAMES = 16
IMAGE_SIZE = 224
FP16 = True


## 1. Setup And Load Model


In [2]:
import os
import sys
import json
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q transformers decord opencv-python-headless tqdm safetensors

import cv2
import numpy as np
import torch
from torchvision.transforms import CenterCrop, Normalize, Resize
from transformers import VideoMAEForVideoClassification
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_dir = Path(DRIVE_MODEL_DIR) if Path(DRIVE_MODEL_DIR).exists() else Path(LOCAL_MODEL_DIR)
print("Using model_dir:", model_dir)

with open(model_dir / "class_names.json", encoding="utf-8") as f:
    class_names = json.load(f)

model = VideoMAEForVideoClassification.from_pretrained(str(model_dir), num_labels=len(class_names))
model.to(device)
if FP16 and device.type == "cuda":
    model.half()
model.eval()

print("Classes:", len(class_names))
print("Device:", device)


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 89.5 MB/s eta 0:00:00
Using model_dir: /content/drive/MyDrive/vsl-recognition/models/videomae_olympic_best


Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

Classes: 100
Device: cpu


## 2. Preprocessing And Prediction Helpers


In [3]:
normalize = Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

def load_video(video_path, num_frames=16):
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(str(video_path), ctx=cpu(0))
        total = len(vr)
        if total >= num_frames:
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
        else:
            indices = np.concatenate([np.arange(total), np.full(num_frames - total, total - 1, dtype=int)])
        return vr.get_batch(indices).asnumpy()
    except Exception:
        cap = cv2.VideoCapture(str(video_path))
        frames = []
        while cap.isOpened():
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        cap.release()
        if not frames:
            raise ValueError(f"Cannot read video: {video_path}")
        frames = np.asarray(frames)
        total = len(frames)
        if total >= num_frames:
            indices = np.linspace(0, total - 1, num_frames, dtype=int)
        else:
            indices = np.concatenate([np.arange(total), np.full(num_frames - total, total - 1, dtype=int)])
        return frames[indices]

def preprocess(frames):
    tensors = []
    for frame in frames:
        tensor = torch.from_numpy(frame).float() / 255.0
        tensor = tensor.permute(2, 0, 1)
        tensor = Resize(IMAGE_SIZE + 32, antialias=True)(tensor)
        tensor = CenterCrop(IMAGE_SIZE)(tensor)
        tensors.append(normalize(tensor))
    video = torch.stack(tensors).unsqueeze(0).to(device)
    if FP16 and device.type == "cuda":
        video = video.half()
    return video

@torch.no_grad()
def predict_video(video_path):
    start = time.time()
    frames = load_video(video_path, NUM_FRAMES)
    video = preprocess(frames)
    logits = model(pixel_values=video).logits[0].float()
    probs = torch.softmax(logits, dim=0)
    top5_probs, top5_idx = torch.topk(probs, min(5, len(class_names)))
    return {
        "label": class_names[top5_idx[0].item()],
        "confidence": top5_probs[0].item(),
        "top5": [(class_names[i.item()], p.item()) for i, p in zip(top5_idx, top5_probs)],
        "latency_ms": (time.time() - start) * 1000,
    }


## 3. Benchmark Latency


In [4]:
@torch.no_grad()
def benchmark(num_runs=50):
    dummy = torch.randn(1, NUM_FRAMES, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    if FP16 and device.type == "cuda":
        dummy = dummy.half()
    for _ in range(5):
        _ = model(pixel_values=dummy)
    if device.type == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in tqdm(range(num_runs), desc="Benchmark"):
        start = time.time()
        _ = model(pixel_values=dummy)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.time() - start) * 1000)
    print(f"Mean latency: {np.mean(times):.1f} ms")
    print(f"P95 latency: {np.percentile(times, 95):.1f} ms")

benchmark(num_runs=50)


Benchmark:   0%|          | 0/50 [00:00<?, ?it/s]

Mean latency: 3565.5 ms
P95 latency: 7250.4 ms


## 4. Test A Validation Video


In [5]:
val_path = Path(METADATA_DIR) / "val.json"
if val_path.exists():
    with open(val_path, encoding="utf-8") as f:
        val_data = json.load(f)
    sample = val_data[0]
    result = predict_video(sample["path"])
    print("Ground truth:", sample.get("class_name"), sample.get("label"))
    print("Prediction:", result)
else:
    print("No val.json found. Skipping sample-video test.")


Ground truth: Vâng lời 86
Prediction: {'label': 'Vâng lời', 'confidence': 0.5668436884880066, 'top5': [('Vâng lời', 0.5668436884880066), ('San sẻ', 0.04133599251508713), ('Bế mạc', 0.015759043395519257), ('Lây bệnh', 0.013054292649030685), ('Kết hôn', 0.01303833071142435)], 'latency_ms': 7826.0345458984375}


## 5. Local Deployment Instructions


In [6]:
print("Drive model folder:")
print(DRIVE_MODEL_DIR)
print("\nDownload/copy that folder to local repo as:")
print(LOCAL_MODEL_DIR)
print("\nExpected local structure:")
print("vsl-recognition/models/videomae_olympic_best/config.json")
print("vsl-recognition/models/videomae_olympic_best/model.safetensors")
print("vsl-recognition/models/videomae_olympic_best/class_names.json")
print("\nRun locally:")
print("streamlit run app.py")


Drive model folder:
/content/drive/MyDrive/vsl-recognition/models/videomae_olympic_best

Download/copy that folder to local repo as:
models/videomae_olympic_best

Expected local structure:
vsl-recognition/models/videomae_olympic_best/config.json
vsl-recognition/models/videomae_olympic_best/model.safetensors
vsl-recognition/models/videomae_olympic_best/class_names.json

Run locally:
streamlit run app.py
